In [7]:
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets,transforms
from torchvision.transforms import ToTensor
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [15]:
traindata=datasets.FashionMNIST(
                            root='datanew',
                            train=True,
                            download=True,
                            transform=ToTensor(),
                            target_transform=None)

testdata=datasets.FashionMNIST(
                            root='datanew',
                            train=False,
                            download=True,
                            transform=ToTensor(),
                            target_transform=None)

                            

In [16]:
from torch.utils.data import DataLoader

In [43]:
newtrain=DataLoader(dataset=traindata,
                   batch_size=32,
                  shuffle=True)
newtest=DataLoader(dataset=testdata,
                   batch_size=32,
                   shuffle=True)


In [20]:
traindata

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: datanew
    Split: Train
    StandardTransform
Transform: ToTensor()

In [24]:
loss_fn=nn.CrossEntropyLoss()

In [77]:
'''model=nn.Sequential(
            nn.Conv2d(in_channels=1,out_channels=10,
                      kernel_size=3,
                      padding=1,
                      stride=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=10,out_channels=10,
                     kernel_size=3,
                     stride=1,
                     padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3),
             nn.Conv2d(in_channels=10,out_channels=10,
                              kernel_size=3,
                              padding=1,
                              stride=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=10,out_channels=10,
                     kernel_size=3,
                     stride=1,
                     padding=1),
            nn.ReLU(),
           nn.MaxPool2d(kernel_size=3),


            nn.Flatten(),
            nn.Linear(in_features=10*7*7,
                      out_features=10)
).to('mps')'''

class convolutional(nn.Module):
    def __init__(self,hidden):
        super().__init__()
        self.layers=nn.Sequential(
                                        nn.Conv2d(in_channels=1,out_channels=hidden,
                                                      kernel_size=3,
                                                      padding=1,
                                                      stride=1),
                                        nn.ReLU(),
                                        nn.Conv2d(in_channels=hidden,out_channels=hidden,
                                                    kernel_size=3,
                                                     padding=1,
                                                     stride=1),
                                        nn.ReLU(),
                                        nn.MaxPool2d(kernel_size=3),
            
                                        nn.Conv2d(in_channels=hidden,out_channels=hidden,
                                                      kernel_size=3,
                                                      padding=1,
                                                      stride=1),
                                        nn.ReLU(),
                                        nn.Conv2d(in_channels=hidden,out_channels=hidden,
                                                    kernel_size=3,
                                                     padding=1,
                                                     stride=1),
                                        nn.ReLU(),
                                        nn.MaxPool2d(kernel_size=3)
        )

        self.find=nn.Sequential(
                        nn.Flatten(),
                        nn.Linear(in_features=hidden*32,
                                    out_features=10)
        )

    def forward(self,image):

        return self.find(self.layers(image))

    
                                            

In [78]:
model=convolutional(10).to('mps')
model

convolutional(
  (layers): Sequential(
    (0): Conv2d(1, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU()
    (7): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU()
    (9): MaxPool2d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
  )
  (find): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=320, out_features=10, bias=True)
  )
)

In [79]:
optimizer=torch.optim.SGD(params=model.parameters(),lr=0.01)

In [80]:
model.state_dict()
model

convolutional(
  (layers): Sequential(
    (0): Conv2d(1, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU()
    (7): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU()
    (9): MaxPool2d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
  )
  (find): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=320, out_features=10, bias=True)
  )
)

In [81]:
def train_step(model:torch.nn.Module,
               data_loader:torch.utils.data.DataLoader,
               loss_fn:torch.nn.Module,
               optimizer:torch.optim.Optimizer,
               device):
    train_loss,train_acc=0,0
    model.to(device)
    for batch,(x,y) in enumerate(data_loader):
        x,y=x.to(device),y.to(device)
        y_pred=model(x)
        loss=loss_fn(y_pred,y)
        train_loss=train_loss+loss
        #train_acc+=accuracy_fn(y_true=y,y_pred=y_pred.argmax(dim=1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss/=len(data_loader)
    train_acc/=len(data_loader)
    print(f"train_loss {train_loss} tacc : {train_acc}")


In [82]:
for i in range(3):
    train_step(model,newtrain,loss_fn,optimizer,'mps')

RuntimeError: linear(): input and weight.T shapes cannot be multiplied (32x90 and 320x10)